# 3.0 — MobileViT smoke test — Food-101

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcoslund/ViT-for-101-food-app/blob/main/notebooks/3.0-mobilevit-smoke-test.ipynb)

Prueba rápida y reproducible del pipeline de fine-tuning de **MobileViT** usando un subconjunto estratificado del `train` y `validation` ya definidos por `2.0-preprocessing.ipynb`.

Esta notebook está pensada para ejecutarse **localmente dentro de la estructura del proyecto**.

**Localmente no hace** (en Colab sí, ver §0):
- `git clone`;
- instalaciones con `pip`;
- descargas automáticas;
- creación de splits nuevos;
- reconstrucción automática del cache;
- evaluación sobre `test`.

**Sí hace:**
- reutiliza `config.py`;
- reutiliza `preprocessing/loaders.py`;
- reutiliza `train_val_split.csv`, `label_map.json` y el cache;
- carga `apple/mobilevit-small` desde `config.MODELS`;
- entrena un subconjunto pequeño para validar GPU, datos, transforms y entrenamiento;
- usa **la misma receta de entrenamiento que `3.1-mobilevit.ipynb`** (learning rate, warmup, early stopping), para que el smoke test valide lo que después corre completo;
- prueba el protocolo de evaluación de `modeling/evaluation.py` (predicciones, FLOPs, latencia) sobre validation, antes de gastar horas en la corrida completa;
- guarda checkpoints bajo `models/mobilevit/smoke_test/`.

> El término *smoke test* evita confundir esta prueba reducida con el **ViT baseline** definido en el README del proyecto.

## 0. Antes de ejecutar

Desde la raíz del repositorio, el entorno local debería estar preparado con:

```bash
uv sync --extra deep
```

y Jupyter debería iniciarse usando ese entorno, por ejemplo:

```bash
uv run jupyter lab
```

Además, el preprocessing debe haberse ejecutado previamente:

```bash
make preprocess
```

o mediante `notebooks/2.0-preprocessing.ipynb`.

**Windows:** el `torch` de PyPI solo usa CPU. Para usar la GPU, después del `uv sync`:

```bash
uv pip install --reinstall torch torchvision --index-url https://download.pytorch.org/whl/cu128
```

y no volver a correr `uv sync` (reinstala el torch de CPU). Abrir Jupyter con el entorno activado (`.venv\Scripts\activate`) en lugar de `uv run`.

La notebook **no reconstruye esos artefactos**: si faltan, se detiene y muestra qué falta.

## 0.1 Parámetros de ejecución

- `USE_DRIVE` (solo Colab): monta Google Drive y guarda ahí el `.tar.gz` del dataset, los checkpoints y los resultados. El disco de Colab se borra al desconectarse; sin Drive, una desconexión a mitad de entrenamiento pierde todo.

Localmente estos parámetros no cambian nada salvo `RESUME`: todo se lee y escribe en las rutas de `config.py`.

In [1]:
# Solo Colab: persistir dataset, checkpoints y resultados en Google Drive.
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/ceia-vpc3"

## 0.2 Entorno (Colab)

Mismo patrón que `2.0-preprocessing.ipynb`: en Colab clona el repo e instala el paquete con el extra `deep`. `torch` ya viene con CUDA en Colab, así que `pip` no lo reinstala. Localmente esta celda solo verifica que el paquete esté instalado.

In [2]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
print(f"Colab: {IN_COLAB}")


def _paquete_disponible():
    try:
        import vit_for_101_food_app  # noqa: F401

        return True
    except ImportError:
        return False


if IN_COLAB:
    REPO_URL = "https://github.com/marcoslund/ViT-for-101-food-app.git"
    REPO_DIR = "/content/ViT-for-101-food-app"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    sys.path.insert(0, REPO_DIR)

    if not _paquete_disponible():
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[deep]"], check=True
        )
    os.chdir(f"{REPO_DIR}/notebooks")

    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
elif not _paquete_disponible():
    raise RuntimeError(
        "vit_for_101_food_app no esta instalado en este kernel. "
        "Elegí el kernel del .venv del proyecto (ver 'Antes de ejecutar')."
    )

print("paquete disponible:", _paquete_disponible())

2026-09-24 15:39:20.393 | INFO     | vit_for_101_food_app.config:<module>:11 - PROJ_ROOT path is: C:\Users\Antonella\Documents\Facultad\Maestría en IA\VC III\Repo proyecto integrador\ViT-for-101-food-app


Colab: False
paquete disponible: True


## 0.3 Datos (Colab)

En Colab el disco arranca vacío en cada sesión, así que acá se hace lo que localmente hace `make preprocess`: descarga (o copia desde Drive) el dataset, genera el split si no está versionado en el repo y arma el cache. El split es determinístico (semilla fija), así que regenerarlo da el mismo `sha256`.

Localmente esta celda no hace nada: los datos tienen que existir de antes.

In [3]:
if IN_COLAB:
    import shutil

    from vit_for_101_food_app import config
    from vit_for_101_food_app import dataset as prep

    tar_local = config.RAW_DATA_DIR / "food-101.tar.gz"
    tar_drive = f"{DRIVE_DIR}/food-101.tar.gz"
    config.RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    if USE_DRIVE and os.path.exists(tar_drive) and not tar_local.exists():
        print("copiando el dataset desde Drive...")
        shutil.copy(tar_drive, tar_local)

    prep.download()  # descarga (~5 GB) y extrae; idempotente
    if USE_DRIVE and not os.path.exists(tar_drive):
        print("guardando el dataset en Drive para la proxima sesion...")
        shutil.copy(tar_local, tar_drive)

    if not config.TRAIN_VAL_SPLIT.exists():
        prep.split()
    prep.cache(workers=os.cpu_count() or 2)  # idempotente

In [4]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Subset
from transformers import (
    AutoModelForImageClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    default_data_collator,
)
from sklearn.metrics import accuracy_score, f1_score

try:
    from vit_for_101_food_app import config
    from vit_for_101_food_app.modeling import evaluation
    from vit_for_101_food_app.preprocessing import cache, loaders, splits
except ImportError as exc:
    raise RuntimeError(
        "No se pudo importar el paquete del proyecto. "
        "Desde la raíz del repo ejecutá `uv sync --extra deep` "
        "y abrí Jupyter con `uv run jupyter lab`."
    ) from exc

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
else:
    print("Dispositivo: CPU")
    print("AVISO: sirve para validar el pipeline, pero será bastante más lento que con GPU.")

c:\Users\Antonella\Documents\Facultad\Maestría en IA\VC III\Repo proyecto integrador\ViT-for-101-food-app\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.11.16
PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM: 8.0 GB


## 1. Verificar artefactos del preprocessing

Los datos deben venir del pipeline ya definido por el proyecto.

Se comprueba que existan:

- `train_val_split.csv`;
- `train_val_split_manifest.json`;
- `label_map.json`;
- cache `food-101-288`;
- `cache_manifest.json`.

No se descarga ni se regenera nada desde esta notebook.

In [5]:
required_files = [
    config.TRAIN_VAL_SPLIT,
    config.TRAIN_VAL_MANIFEST,
    config.LABEL_MAP,
    config.CACHE_DIR / "cache_manifest.json",
]

missing = [Path(p) for p in required_files if not Path(p).exists()]

if missing:
    missing_txt = "\n".join(f" - {p}" for p in missing)
    raise FileNotFoundError(
        "Faltan artefactos del preprocessing:\n"
        f"{missing_txt}\n\n"
        "Ejecutá primero `make preprocess` desde la raíz del proyecto "
        "o corré `notebooks/2.0-preprocessing.ipynb`."
    )

if not cache.cache_is_valid(
    cache_dir=config.CACHE_DIR,
    short_side=config.CACHE_SHORT_SIDE,
):
    raise RuntimeError(
        f"El cache en {config.CACHE_DIR} existe pero no es válido para "
        f"CACHE_SHORT_SIDE={config.CACHE_SHORT_SIDE}. Regeneralo con `make cache`."
    )

id2label, label2id = splits.load_label_map(config.LABEL_MAP)
NUM_LABELS = len(label2id)

print("Preprocessing OK")
print("train/val split:", config.TRAIN_VAL_SPLIT)
print("label map:", config.LABEL_MAP)
print("cache:", config.CACHE_DIR)
print("clases:", NUM_LABELS)

assert NUM_LABELS == 101, f"Se esperaban 101 clases y se encontraron {NUM_LABELS}."

Preprocessing OK
train/val split: C:\Users\Antonella\Documents\Facultad\Maestría en IA\VC III\Repo proyecto integrador\ViT-for-101-food-app\data\processed\train_val_split.csv
label map: C:\Users\Antonella\Documents\Facultad\Maestría en IA\VC III\Repo proyecto integrador\ViT-for-101-food-app\data\processed\label_map.json
cache: C:\Users\Antonella\Documents\Facultad\Maestría en IA\VC III\Repo proyecto integrador\ViT-for-101-food-app\data\interim\food-101-288
clases: 101


## 2. DataLoaders de MobileViT

Se usa `loaders.build_dataloaders("mobilevit", ...)`.

La notebook **no reimplementa** resize, crop, BGR/RGB, escala ni augmentation. Todo eso queda bajo responsabilidad de los módulos de preprocessing ya verificados en el proyecto.

In [6]:
# Para la RTX 5060 Laptop de 8 GB, 16 es un punto de partida razonable.
# Si aparece CUDA out of memory, bajar a 8.
# En una GPU con >= 12 GB se puede probar 32.
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    BATCH_SIZE = 32 if gpu_mem_gb >= 12 else 16
else:
    BATCH_SIZE = 8

NUM_WORKERS = min(4, os.cpu_count() or 1)  # Colab tiene 2 CPUs

dls = loaders.build_dataloaders(
    "mobilevit",
    train_policy="standard",
    source="cache",
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    splits_to_load=("train", "val"),
    images_root=config.CACHE_DIR,
    csv_path=config.TRAIN_VAL_SPLIT,
    label_map_path=config.LABEL_MAP,
    meta_dir=config.FOOD101_META_DIR,
)

train_ds_full = dls["train"].dataset
val_ds_full = dls["val"].dataset

print("batch size:", BATCH_SIZE)
print("train completo:", len(train_ds_full))
print("validation completo:", len(val_ds_full))

batch = next(iter(dls["train"]))
print("pixel_values:", tuple(batch["pixel_values"].shape), batch["pixel_values"].dtype)
print("labels:", tuple(batch["labels"].shape))
print(
    "rango pixel_values:",
    float(batch["pixel_values"].min()),
    float(batch["pixel_values"].max()),
)

batch size: 16
train completo: 68175
validation completo: 7575
pixel_values: (16, 3, 256, 256) torch.float32
labels: (16,)
rango pixel_values: 0.0 1.0


## 3. Subconjunto estratificado para el smoke test

No se crea un nuevo split.

Se toman índices **dentro del train y validation ya fijados por preprocessing**, manteniendo todas las 101 clases:

- 50 imágenes por clase en train → **5.050 imágenes**;
- 10 imágenes por clase en validation → **1.010 imágenes**.

`test` no se usa en esta notebook.

In [7]:
TRAIN_PER_CLASS = 50
VAL_PER_CLASS = 10
SEED = int(config.SEED)

train_frame = splits.load_split(
    "train",
    csv_path=config.TRAIN_VAL_SPLIT,
    meta_dir=config.FOOD101_META_DIR,
)
val_frame = splits.load_split(
    "val",
    csv_path=config.TRAIN_VAL_SPLIT,
    meta_dir=config.FOOD101_META_DIR,
)

def stratified_indices(frame: pd.DataFrame, n_per_class: int, seed: int) -> list[int]:
    sampled = (
        frame.groupby("class_dir", group_keys=False)
        .sample(n=n_per_class, random_state=seed)
        .sort_index()
    )
    return sampled.index.tolist()

train_idx = stratified_indices(train_frame, TRAIN_PER_CLASS, SEED)
val_idx = stratified_indices(val_frame, VAL_PER_CLASS, SEED)

train_ds = Subset(train_ds_full, train_idx)
val_ds = Subset(val_ds_full, val_idx)

print("smoke train:", len(train_ds))
print("smoke val:", len(val_ds))
print("clases train:", train_frame.iloc[train_idx]["class_dir"].nunique())
print("clases val:", val_frame.iloc[val_idx]["class_dir"].nunique())

assert len(train_ds) == 101 * TRAIN_PER_CLASS
assert len(val_ds) == 101 * VAL_PER_CLASS

smoke train: 5050
smoke val: 1010
clases train: 101
clases val: 101


## 4. MobileViT preentrenado

El checkpoint se obtiene del registry central del proyecto:

```python
config.MODELS["mobilevit"]
```

La cabeza de clasificación se adapta a las 101 clases de Food-101.

In [8]:
CHECKPOINT = config.MODELS["mobilevit"]
print("checkpoint:", CHECKPOINT)

model = AutoModelForImageClassification.from_pretrained(
    CHECKPOINT,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

params_total = sum(p.numel() for p in model.parameters())
params_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"parámetros totales: {params_total / 1e6:.2f} M")
print(f"parámetros entrenables: {params_trainable / 1e6:.2f} M")

checkpoint: apple/mobilevit-small


[transformers] You passed `num_labels=101` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 347/347 [00:00<00:00, 38548.14it/s]
[transformers] MobileViTForImageClassification LOAD REPORT from: apple/mobilevit-small
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 640]) vs model:torch.Size([101, 640])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([101])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


parámetros totales: 5.00 M
parámetros entrenables: 5.00 M


## 5. Métricas

In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
    }

## 6. Configuración del entrenamiento

Misma receta que `3.1-mobilevit.ipynb`; solo cambian los datos (el subconjunto) y el máximo de épocas.

- **10 épocas como máximo**, early stopping con paciencia 3: 10 es un techo, no una obligación.
- Mejor checkpoint por **F1 macro** en validation.
- **Warmup lineal** durante el primer 5 % de los pasos: la cabeza de clasificación arranca aleatoria y los primeros gradientes, grandes, no deberían llegar con el learning rate completo a un backbone preentrenado.
- `LEARNING_RATE = 5e-4` es alto para fine-tuning (lo habitual ronda `5e-5`). Queda así porque el backbone es chico y el smoke test aprende con este valor, pero **no está elegido con evidencia**: si se cambia, cambiarlo en las dos notebooks y en la de ViT.

In [10]:
OUTPUT_DIR = config.MODELS_DIR / "mobilevit" / "smoke_test"
if IN_COLAB and USE_DRIVE:
    OUTPUT_DIR = Path(DRIVE_DIR) / "models" / "mobilevit" / "smoke_test"

EPOCHS = 10
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01
WARMUP = 0.05  # fracción de los pasos totales
EARLY_STOPPING_PATIENCE = 3

# El smoke test es descartable: cada corrida empieza de cero.
import shutil

shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("output:", OUTPUT_DIR)
print("epochs máx.:", EPOCHS)
print("learning rate:", LEARNING_RATE)
print("warmup:", WARMUP)
print("best model metric: f1_macro")

output: C:\Users\Antonella\Documents\Facultad\Maestría en IA\VC III\Repo proyecto integrador\ViT-for-101-food-app\models\mobilevit\smoke_test
epochs máx.: 10
learning rate: 0.0005
warmup: 0.05
best model metric: f1_macro


In [11]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=NUM_WORKERS,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE
        )
    ],
)

## 7. Entrenamiento

In [12]:
t0 = time.time()
train_result = trainer.train()
elapsed = time.time() - t0

print(f"\nTiempo total: {elapsed / 60:.1f} min = {elapsed / 3600:.2f} h")
print("best checkpoint:", trainer.state.best_model_checkpoint)
print("best F1 macro:", trainer.state.best_metric)

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,3.546368,3.222013,0.282178,0.231537,0.231537
2,2.596699,2.377485,0.419802,0.376272,0.376272
3,2.051301,1.960506,0.511881,0.485170,0.485170
4,1.601849,1.814156,0.547525,0.528601,0.528601
5,1.309218,1.796992,0.538614,0.522491,0.522491
6,0.987311,1.734907,0.581188,0.571839,0.571839
7,0.753136,1.608686,0.608911,0.601204,0.601204
8,0.613275,1.679898,0.600990,0.593431,0.593431
9,0.446758,1.673428,0.610891,0.598585,0.598585
10,0.375338,1.710479,0.618812,0.608229,0.608229


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 29.41it/s]



Tiempo total: 14.4 min = 0.24 h
best checkpoint: C:\Users\Antonella\Documents\Facultad\Maestría en IA\VC III\Repo proyecto integrador\ViT-for-101-food-app\models\mobilevit\smoke_test\checkpoint-3160
best F1 macro: 0.6082294065519558


## 8. Evaluación del mejor checkpoint sobre validation

`Trainer` ya recarga el mejor checkpoint al finalizar porque `load_best_model_at_end=True`.

In [13]:
val_metrics = trainer.evaluate(val_ds)
val_metrics

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,F1 Weighted
0.375338,1.710479,10,0.618812,0.608229,0.608229


{'eval_loss': 1.7104787826538086,
 'eval_accuracy': 0.6188118811881188,
 'eval_f1_macro': 0.6082294065519558,
 'eval_f1_weighted': 0.6082294065519557}

## 9. Curva de métricas por época

In [14]:
history = pd.DataFrame(trainer.state.log_history)

epoch_metrics = history[
    history["eval_loss"].notna()
][
    [
        "epoch",
        "eval_loss",
        "eval_accuracy",
        "eval_f1_macro",
        "eval_f1_weighted",
    ]
].copy()

epoch_metrics

,epoch,eval_loss,eval_accuracy,eval_f1_macro,eval_f1_weighted
6,1.0,3.222013,0.282178,0.231537,0.231537
13,2.0,2.377485,0.419802,0.376272,0.376272
20,3.0,1.960506,0.511881,0.485170,0.485170
28,4.0,1.814156,0.547525,0.528601,0.528601
35,5.0,1.796992,0.538614,0.522491,0.522491
42,6.0,1.734907,0.581188,0.571839,0.571839
50,7.0,1.608686,0.608911,0.601204,0.601204
57,8.0,1.679898,0.600990,0.593431,0.593431
64,9.0,1.673428,0.610891,0.598585,0.598585
72,10.0,1.710479,0.618812,0.608229,0.608229


## 10. Guardar el mejor modelo

In [15]:
BEST_DIR = OUTPUT_DIR / "best"
trainer.save_model(str(BEST_DIR))

print("Modelo guardado en:", BEST_DIR)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 40.00it/s]

Modelo guardado en: C:\Users\Antonella\Documents\Facultad\Maestría en IA\VC III\Repo proyecto integrador\ViT-for-101-food-app\models\mobilevit\smoke_test\best


## 11. Prueba del protocolo de evaluación

La corrida completa (`3.1`) evalúa sobre `test` con `vit_for_101_food_app.modeling.evaluation`. Acá se ejercita **ese mismo código sobre validation**, para que un error aparezca en minutos y no después de horas de entrenamiento.

`test` sigue sin usarse: del subset del benchmark solo se verifica que el archivo esté intacto (hash) y que todas sus imágenes existan en el split de test.

In [16]:
val_output = trainer.predict(val_ds)
val_preds = evaluation.predictions_frame(
    val_frame.iloc[val_idx].reset_index(drop=True), val_output.predictions, id2label
)
pd.Series(evaluation.split_metrics(val_preds))

n                1010.000000
n_clases          101.000000
accuracy            0.618812
top5_accuracy       0.831683
f1_macro            0.608229
f1_weighted         0.608229
dtype: float64

In [17]:
subset = evaluation.load_benchmark_subset()
test_frame = splits.load_split("test", csv_path=config.TRAIN_VAL_SPLIT, meta_dir=config.FOOD101_META_DIR)
faltan = set(subset["rel"]) - set(test_frame["rel"])

print("subset del benchmark:", len(subset), "imágenes, hash OK")
print("terciles:", subset["tercil"].value_counts().to_dict())
assert not faltan, f"{len(faltan)} imágenes del subset no están en test"

subset del benchmark: 2525 imágenes, hash OK
terciles: {'dificil': 850, 'facil': 850, 'medio': 825}


In [18]:
sample = val_ds[0]["pixel_values"].unsqueeze(0)
device = trainer.args.device

print("costo por imagen:", evaluation.count_flops(model, sample.to(device)))
print("latencia", device, evaluation.measure_latency(model, sample, device, n_runs=20))
print("latencia cpu", evaluation.measure_latency(model, sample, "cpu", n_runs=10))

W0924 15:55:37.049000 22028 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


costo por imagen: {'gflops': 4.000512256, 'gmacs': 2.000256128}
latencia cuda:0 {'device': 'NVIDIA GeForce RTX 5060 Laptop GPU', 'batch_size': 1, 'cpu_threads': 1, 'n_runs': 20, 'median_ms': 11.583700000301178, 'p90_ms': 16.232179999133223, 'mean_ms': 12.358245000086754}
latencia cpu {'device': 'cpu', 'batch_size': 1, 'cpu_threads': 1, 'n_runs': 10, 'median_ms': 71.37530000090919, 'p90_ms': 104.65387000058399, 'mean_ms': 79.42278000045917}


## Interpretación del smoke test

Esta corrida **no es la evaluación final de MobileViT en Food-101**. Usa solo una fracción del train y validation.

Sirve para comprobar:

- que los artefactos de preprocessing son compatibles;
- que los transforms de MobileViT funcionan;
- que el modelo aprende;
- que las métricas y checkpoints se generan correctamente;
- que el protocolo de evaluación (predicciones, subset del benchmark, FLOPs, latencia) corre sin errores;

El conjunto `test` queda reservado para el experimento final.